In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from mlxtend.regressor import StackingCVRegressor
from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split


2024-09-20 15:21:31.773412: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-09-20 15:21:31.773546: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-20 15:21:31.954000: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


numpy:精度计算
pandas：读取CSV文件
KFold, train_test_split：测试数据集划分
load_img,img_to_array：读取图片文件，将图片转为数组
Conv2D：卷积
MaxPooling2D:池化层
Flatten：把提取特征转化为可被计算的权重
Dense：致密层，转矩阵
Sequential：序列化
load_model：提供API接口
Adam：提高学习率
MeanAbsoluteError：残差评估

In [2]:
# Load the dataset
data = pd.read_csv('/kaggle/input/new-sugar-analysis1/dataset.csv')

# Split the data into features (images) and targets (luminance)
X = []
y = data['色泽IU'].values




锐化图像，调整对比度//数据预处理
图片数据增强，随机裁剪/旋转
图片读入构建新的数据集//直接加RGB进来


In [3]:
# def calculate_average_rgb(image_path):
#     # 加载图片并转换为 RGB
#     img = Image.open(image_path)
#     img = img.convert('RGB')
    
#     # 将图片数据转换为 numpy 数组
#     img_array = np.array(img)
    
#     # 计算平均 RGB 值
#     average_rgb = img_array.mean(axis=(0, 1))
    
#     # 返回平均 RGB 值
#     return average_rgb


In [4]:
for idx, image_path in enumerate(data['IMG_Name']):
    img = load_img("/kaggle/input/sugar-analysis/newdata/"+image_path, target_size=(1024, 1024))
    img_array = img_to_array(img)
    average_rgb = img_array.mean(axis=(0, 1))
#     X.append([data.iloc[idx]['干燥失重g/100g'],data.iloc[idx]['还原糖分g/100g'],data.iloc[idx]['电导灰分g/100g'],data.iloc[idx]['浑浊度MAU'],data.iloc[idx]['不溶于水杂质mg/kg'],data.iloc[idx]['粒径'],data.iloc[idx]['糖分'],average_rgb[0],average_rgb[1],average_rgb[2]])
    X.append([data.iloc[idx]['电导灰分g/100g'],data.iloc[idx]['干燥失重g/100g'],data.iloc[idx]['粒径'],average_rgb[0],average_rgb[1],average_rgb[2]])

X = np.array(X)
# print(x_o)

# Define the k-fold cross-validation
n_splits = 5

# Initialize the validation scores
validation_scores = []



下一步优化方向：target_size 224/512/768/1024


In [5]:
print(X.shape)

(477, 6)


In [6]:
kfolds = 5
#X_img_train , X_img_val = X[train_index],X[val_index]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
# X_train , X_val = X[train_index],X[val_index]
# y_train, y_val = y[train_index], y[val_index]
xgboost= XGBRegressor(n_estimators=1000, learning_rate=0.05, n_jobs=4)
lightgbm = LGBMRegressor(objective='regression', 
                                   num_leaves=4,
                                   learning_rate=0.01, 
                                   n_estimators=5000,
                                   max_bin=200, 
                                   bagging_fraction=0.75,
                                   bagging_freq=5, 
                                   bagging_seed=7,
                                   feature_fraction=0.2,
                                   feature_fraction_seed=7,
                                   verbose=-1,
                                   )
gbr = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05, max_depth=4, max_features='sqrt', min_samples_leaf=15, min_samples_split=10, loss='huber', random_state =42)     
alphas_alt = [14.5, 14.6, 14.7, 14.8, 14.9, 15, 15.1, 15.2, 15.3, 15.4, 15.5]
alphas2 = [5e-05, 0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007, 0.0008]
e_alphas = [0.0001, 0.0002, 0.0003, 0.0004, 0.0005, 0.0006, 0.0007]
e_l1ratio = [0.8, 0.85, 0.9, 0.95, 0.99, 1]
ridge = make_pipeline(RobustScaler(),
                      RidgeCV(alphas=alphas_alt, cv=kfolds))
lasso = make_pipeline(RobustScaler(),
                      LassoCV(max_iter=int(1e7), alphas=alphas2,
                              random_state=42, cv=kfolds))
elasticnet = make_pipeline(RobustScaler(),
                           ElasticNetCV(max_iter=int(1e7), alphas=e_alphas,
                                        cv=kfolds, l1_ratio=e_l1ratio))
stack_gen = StackingCVRegressor(regressors=(ridge, lasso, elasticnet, gbr, xgboost, lightgbm),
                            meta_regressor=xgboost,
                            use_features_in_secondary=True)
stack_gen.fit(X_train, y_train)
predictions = stack_gen.predict(X_val)
xgboost.fit(X_train,y_train)
lightgbm.fit(X_train,y_train)
gbr.fit(X_train,y_train)
ridge.fit(X_train,y_train)
lasso.fit(X_train,y_train)
elasticnet.fit(X_train,y_train)


Pipeline(steps=[('robustscaler', RobustScaler()),
                ('elasticnetcv',
                 ElasticNetCV(alphas=[0.0001, 0.0002, 0.0003, 0.0004, 0.0005,
                                      0.0006, 0.0007],
                              cv=5, l1_ratio=[0.8, 0.85, 0.9, 0.95, 0.99, 1],
                              max_iter=10000000))])

In [7]:
print("xgb:"+str(mean_absolute_error(xgboost.predict(X_val),y_val)))
print("lgbm:"+str(mean_absolute_error(lightgbm.predict(X_val),y_val)))
print("gbr:"+str(mean_absolute_error(gbr.predict(X_val),y_val)))
print("ridge:"+str(mean_absolute_error(ridge.predict(X_val),y_val)))
print("lasso:"+str(mean_absolute_error(lasso.predict(X_val),y_val)))
print("elasticnet:"+str(mean_absolute_error(elasticnet.predict(X_val),y_val)))
print("Mean Absolute Error: " + str(mean_absolute_error(predictions, y_val)))

xgb:4.764573276042938
lgbm:8.659089723492505
gbr:5.908117845316214
ridge:7.92332535741466
lasso:7.039109587208598
elasticnet:7.039005946286964
Mean Absolute Error: 5.945345942179362


Stack 4.62

引入残差神经网络/Resnet
加入多层卷积核
引入backbone层/可选

In [8]:
image_path = "354_1.jpg"
idx=100
# 加载所有模型
img = load_img("/kaggle/input/sugar-analysis/newdata/"+image_path, target_size=(1024, 1024))
img_array = img_to_array(img)
average_rgb = img_array.mean(axis=(0, 1))
P=[data.iloc[idx]['干燥失重g/100g'],data.iloc[idx]['还原糖分g/100g'],data.iloc[idx]['电导灰分g/100g'],data.iloc[idx]['浑浊度MAU'],data.iloc[idx]['不溶于水杂质mg/kg'],data.iloc[idx]['粒径'],data.iloc[idx]['糖分'],average_rgb[0],average_rgb[1],average_rgb[2]]
P=np.array(P).reshape(1,-1)
print(P)

print(xgboost.predict(P))

#/kaggle/input/sugar-analysis/newdata/094_1.jpg


[[3.90000000e-02 1.20000000e-02 4.60000000e-02 4.20000000e+01
  6.00000000e+00 8.50000000e-01 9.90000000e-01 1.58010361e+02
  1.65387573e+02 6.37350464e+01]]


ValueError: Feature shape mismatch, expected: 6, got 10